# Test Suite — COW vs MOR (workers persistentes)

2 workers se inicializan **una sola vez** y reciben todas las tareas sin reiniciar.

**Run All** — tarda ~2-3 min en vez de 15-20 min.

In [ ]:
!pip install pyspark
!sudo apt-get update && sudo apt-get install -y openjdk-17-jdk 
import subprocess, sys, os, threading, json, glob, time
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime

CATALOG         = "players"
DATABASE        = "concurrency_tests"
TARGET          = f"{CATALOG}.{DATABASE}.concurrent_table"
CONFLICT_ROW_ID = 7
NUM_ITERATIONS  = 5   # repeticiones por escenario

SCENARIOS = [
    ("s01_ins_ins",  "insert  vs insert",  ("insert", "0"),                  ("insert", "0"),                  "✅✅ sin conflicto"),
    ("s02_ins_read", "insert  vs read",    ("insert", "0"),                  ("read",   "0"),                  "✅✅ sin conflicto"),
    ("s03_ins_upd",  "insert  vs update",  ("insert", "0"),                  ("update", str(CONFLICT_ROW_ID)), "✅✅ sin conflicto"),
    ("s04_ins_del",  "insert  vs delete",  ("insert", "0"),                  ("delete", str(CONFLICT_ROW_ID)), "✅✅ sin conflicto"),
    ("s05_ins_mrg",  "insert  vs merge",   ("insert", "0"),                  ("merge",  str(CONFLICT_ROW_ID)), "✅✅ sin conflicto"),
    ("s06_read_read","read    vs read",    ("read",   "0"),                  ("read",   "0"),                  "✅✅ sin conflicto"),
    ("s07_read_upd", "read    vs update",  ("read",   "0"),                  ("update", str(CONFLICT_ROW_ID)), "✅✅ sin conflicto"),
    ("s08_upd_upd",  "update  vs update",  ("update", str(CONFLICT_ROW_ID)), ("update", str(CONFLICT_ROW_ID)), "⚡ conflicto"),
    ("s09_upd_del",  "update  vs delete",  ("update", str(CONFLICT_ROW_ID)), ("delete", str(CONFLICT_ROW_ID)), "⚡ conflicto"),
    ("s10_upd_mrg",  "update  vs merge",   ("update", str(CONFLICT_ROW_ID)), ("merge",  str(CONFLICT_ROW_ID)), "⚡ conflicto"),
    ("s11_del_del",  "delete  vs delete",  ("delete", str(CONFLICT_ROW_ID)), ("delete", str(CONFLICT_ROW_ID)), "⚡ conflicto"),
    ("s12_del_mrg",  "delete  vs merge",   ("delete", str(CONFLICT_ROW_ID)), ("merge",  str(CONFLICT_ROW_ID)), "⚡ conflicto"),
    ("s13_mrg_mrg",  "merge   vs merge",   ("merge",  str(CONFLICT_ROW_ID)), ("merge",  str(CONFLICT_ROW_ID)), "⚡ conflicto"),
]

WRITE_MODES = [
    ("copy-on-write", "COW"),
    ("merge-on-read", "MOR"),
]

print(f"Escenarios: {len(SCENARIOS)} × modos: {len(WRITE_MODES)} × iter: {NUM_ITERATIONS}")
print(f"SparkSessions necesarias: 2 (una por worker, para siempre)")

In [ ]:
# ── SPARK DEL NOTEBOOK (setup de tablas) ───────────────────
def get_spark():
    builder = SparkSession.builder.appName("test_suite_notebook")
    builder = builder.config("spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1,"
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.367")
    builder = builder.config("spark.jars.excludes",
        "org.apache.hadoop:hadoop-client-runtime,org.apache.hadoop:hadoop-client-api")
    builder = builder.config("spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    for k, v in [
        ("spark.sql.catalog.players",                            "org.apache.iceberg.spark.SparkCatalog"),
        ("spark.sql.catalog.players.type",                       "rest"),
        ("spark.sql.catalog.players.uri",                        "http://172.16.58.11:32688"),
        ("spark.sql.catalog.players.warehouse",                  "s3://warehouse/"),
        ("spark.sql.catalog.players.io-impl",                    "org.apache.iceberg.aws.s3.S3FileIO"),
        ("spark.sql.catalog.players.s3.endpoint",                "http://172.16.58.11:31224"),
        ("spark.sql.catalog.players.s3.region",                  "us-east-1"),
        ("spark.sql.catalog.players.s3.path-style-access",       "true"),
        ("spark.sql.catalog.players.client.region",              "us-east-1"),
        ("spark.hadoop.fs.s3a.endpoint",                         "http://172.16.58.11:31224"),
        ("spark.hadoop.fs.s3a.endpoint.region",                  "us-east-1"),
        ("spark.hadoop.fs.s3a.access.key",                       "GK5f421d5f440758f74b0e0312"),
        ("spark.hadoop.fs.s3a.secret.key",                       "409baa63477885db12cd1db0a518748c5e83e971b5e8cf2129fe6c7498de125d"),
        ("spark.hadoop.fs.s3a.path.style.access",                "true"),
        ("spark.hadoop.fs.s3a.impl",                             "org.apache.hadoop.fs.s3a.S3AFileSystem"),
        ("spark.sql.catalog.players.s3.checksum-algorithm",      "NONE"),
        ("spark.sql.catalog.players.s3.streaming-upload-enabled","false"),
        ("spark.sql.catalog.players.s3.chunked-encoding-enabled","false"),
        ("spark.sql.catalog.players.s3.payload-signing-enabled", "true"),
        ("spark.sql.catalog.players.s3.http-client-type",        "urlconnection"),
        ("spark.executor.extraJavaOptions",                      "-Daws.requestChecksumCalculation=when_required"),
        ("spark.driver.extraJavaOptions",                        "-Daws.requestChecksumCalculation=when_required"),
        ("spark.sql.catalog.players.s3.access-key-id",          "GK5f421d5f440758f74b0e0312"),
        ("spark.sql.catalog.players.s3.secret-access-key",      "409baa63477885db12cd1db0a518748c5e83e971b5e8cf2129fe6c7498de125d"),
    ]:
        builder = builder.config(k, v)
    return builder.getOrCreate()

spark = get_spark()
spark.sparkContext.setLogLevel("ERROR")

schema = StructType([
    StructField("id",     IntegerType()),
    StructField("source", StringType()),
    StructField("value",  StringType()),
    StructField("ts",     TimestampType()),
])

def setup_table(write_mode):
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {CATALOG}.{DATABASE}")
    spark.sql(f"DROP TABLE IF EXISTS {TARGET}")
    spark.sql(f"""
        CREATE TABLE {TARGET} (
            id INT, source STRING, value STRING, ts TIMESTAMP
        ) USING iceberg
        TBLPROPERTIES (
            'write.update.mode'        = '{write_mode}',
            'write.merge.mode'         = '{write_mode}',
            'write.delete.mode'        = '{write_mode}',
            'commit.retry.num-retries' = '4',
            'commit.retry.min-wait-ms' = '100',
            'commit.retry.max-wait-ms' = '2000'
        )
    """)
    seed = [(i, "seed", f"val_{i}", datetime.now()) for i in range(1, 21)]
    spark.createDataFrame(seed, schema).writeTo(TARGET).append()

def reset_conflict_row():
    spark.catalog.refreshTable(TARGET)
    if spark.table(TARGET).filter(f"id = {CONFLICT_ROW_ID}").count() == 0:
        spark.createDataFrame(
            [(CONFLICT_ROW_ID, "seed", f"val_{CONFLICT_ROW_ID}", datetime.now())], schema
        ).writeTo(TARGET).append()

print("✅ SparkSession del notebook lista")

In [ ]:
# ── ARRANCAR WORKERS PERSISTENTES ─────────────────────────
# Solo ocurre UNA vez — aquí está todo el coste de inicialización

clean_env = {k: v for k, v in os.environ.items()
             if "SPARK" not in k and "PYSPARK" not in k}

# Limpia ficheros anteriores
for f in glob.glob("/tmp/worker_*"):
    try: os.remove(f)
    except: pass

processes = []
for i in range(2):
    proc = subprocess.Popen(
        [sys.executable, "worker.py", str(i)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env=clean_env
    )
    processes.append(proc)
    print(f"Worker {i} arrancado — PID {proc.pid}")

# Hilo para ver logs de los workers en tiempo real
def log_worker(proc, wid):
    for line in proc.stdout:
        line = line.strip()
        if line and "INFO:" in line:
            print(f"  [W{wid}] {line}", flush=True)

for i, proc in enumerate(processes):
    t = threading.Thread(target=log_worker, args=(proc, i), daemon=True)
    t.start()

# Espera a que los dos tengan Spark listo
print("\nEsperando inicialización de Spark (solo esta vez)...")
t0 = time.time()
while True:
    ready = [i for i in range(2) if os.path.exists(f"/tmp/worker_{i}_ready")]
    print(f"  {len(ready)}/2 listos...", end="\r", flush=True)
    if len(ready) == 2:
        break
    if time.time() - t0 > 300:
        raise TimeoutError("Workers no arrancaron")
    time.sleep(1)

print(f"\n✅ Ambos workers listos en {time.time()-t0:.1f}s — no volverán a inicializar")

In [ ]:
# ── FUNCIONES DE COORDINACIÓN ──────────────────────────────
POLL_INTERVAL_S = 0.05
WORKER_READY_TIMEOUT_S = 180

def send_task(worker_id, operation, target, scenario, iteration):
    """Escribe una tarea para el worker y espera su resultado."""
    task_file   = f"/tmp/worker_{worker_id}_task.json"
    result_file = f"/tmp/worker_{worker_id}_result.json"
    result_done = f"/tmp/worker_{worker_id}_result_done"

    # Limpia resultado anterior
    for f in [result_file, result_done]:
        try: os.remove(f)
        except: pass

    # Escribe la tarea
    with open(task_file, "w") as f:
        json.dump({"operation": operation, "target": target,
                   "scenario": scenario, "iteration": iteration}, f)

    # Espera resultado
    t0 = time.time()
    while not os.path.exists(result_done):
        if time.time() - t0 > 120:
            raise TimeoutError(f"Worker {worker_id} no respondió")
        time.sleep(0.02)

    with open(result_file) as f:
        result = json.load(f)

    os.remove(result_done)
    return result


def run_scenario_concurrent(scenario_id, w0_op, w0_target, w1_op, w1_target, iteration):
    """
    Envía tareas a los dos workers al mismo tiempo usando threads.
    Los dos send_task se ejecutan en paralelo — máxima presión sobre Iceberg.
    """
    results = [None, None]

    def task(wid, op, target):
        results[wid] = send_task(wid, op, target, scenario_id, iteration)

    t0 = threading.Thread(target=task, args=(0, w0_op, w0_target))
    t1 = threading.Thread(target=task, args=(1, w1_op, w1_target))

    # Arranca los dos a la vez — aquí los threads son solo I/O (escritura de fichero)
    # el trabajo pesado ocurre en los procesos worker
    t0.start(); t1.start()
    t0.join();  t1.join()

    return results


def stop_workers():
    for i in range(2):
        task_file = f"/tmp/worker_{i}_task.json"
        with open(task_file, "w") as f:
            json.dump({"operation": "stop"}, f)
    for proc in processes:
        proc.wait(timeout=10)
    print("✅ Workers detenidos")

print("✅ Funciones de coordinación listas")

def _extract_failed_worker_ids(error):
    """Extrae ids de worker de cualquier texto que contenga patrones W<id>:."""
    msg = str(error)
    ids = []
    i = 0
    while i < len(msg):
        if msg[i] == "W":
            j = i + 1
            while j < len(msg) and msg[j].isdigit():
                j += 1
            if j > i + 1 and j < len(msg) and msg[j] == ":":
                ids.append(int(msg[i + 1:j]))
                i = j + 1
                continue
        i += 1
    return sorted(set(ids))

def _cleanup_worker_ipc(worker_id):
    for f in [
        f"/tmp/worker_{worker_id}_task.json",
        f"/tmp/worker_{worker_id}_result.json",
        f"/tmp/worker_{worker_id}_result_done",
        f"/tmp/worker_{worker_id}_ready",
    ]:
        try:
            os.remove(f)
        except FileNotFoundError:
            pass
        except Exception:
            pass

def _ensure_worker_logging(proc, worker_id):
    t = threading.Thread(target=log_worker, args=(proc, worker_id), daemon=True)
    t.start()


def restart_worker(worker_id, reason="unknown"):
    """Reinicia un worker concreto y espera a que vuelva a estar ready."""
    print(f"  ↻ Reiniciando worker {worker_id} ({reason})")

    if worker_id < len(processes):
        old_proc = processes[worker_id]
        if old_proc.poll() is None:
            try:
                old_proc.terminate()
                old_proc.wait(timeout=8)
            except Exception:
                try:
                    old_proc.kill()
                except Exception:
                    pass

    _cleanup_worker_ipc(worker_id)

    proc = subprocess.Popen(
        [sys.executable, "worker.py", str(worker_id)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=clean_env,
    )

    if worker_id < len(processes):
        processes[worker_id] = proc
    else:
        processes.append(proc)

    _ensure_worker_logging(proc, worker_id)

    ready_file = f"/tmp/worker_{worker_id}_ready"
    t0 = time.time()
    while not os.path.exists(ready_file):
        if proc.poll() is not None:
            raise RuntimeError(f"Worker {worker_id} murio al reiniciar (exit={proc.poll()})")
        if time.time() - t0 > WORKER_READY_TIMEOUT_S:
            raise TimeoutError(f"Worker {worker_id} no quedo ready tras reinicio")
        time.sleep(POLL_INTERVAL_S)

In [ ]:
# ── EJECUTAR TODOS LOS ESCENARIOS EN AMBOS MODOS ──────────

all_results = []
all_summary = []
NEEDS_RESET = {"update", "delete", "merge"}

for write_mode, mode_tag in WRITE_MODES:
    print(f"\n{'═'*60}")
    print(f"  MODO: {write_mode} ({mode_tag})")
    print(f"{'═'*60}")

    setup_table(write_mode)
    print(f"  Tabla recreada en modo {write_mode}")

    for scenario_id, desc, w0, w1, expected in SCENARIOS:
        w0_op, w0_target = w0
        w1_op, w1_target = w1

        sc_results = []
        t_sc = time.time()

        for iteration in range(NUM_ITERATIONS):
            # Restaura la fila de conflicto si fue borrada
            if w0_op in NEEDS_RESET or w1_op in NEEDS_RESET:
                reset_conflict_row()

            attempt = 0
            while True:
                try:
                    pair = run_scenario_concurrent(
                        scenario_id, w0_op, w0_target, w1_op, w1_target, iteration
                    )
                    break
                except Exception as e:
                        failed_workers = _extract_failed_worker_ids(e)
                        can_retry = attempt < MAX_ITER_RETRIES and len(failed_workers) > 0

                        if can_retry:
                            print(f"  ⚠️ {scenario_id} iter={iteration} intento={attempt+1}: {e}")
                            for wid in failed_workers:
                                restart_worker(wid, reason=f"{scenario_id}/iter={iteration}")
                            attempt += 1
                            continue

                        print(f"  ❌ {scenario_id} iter={iteration}: {e}")
                        raise
            for r in pair:
                if r:
                    r["write_mode"] = mode_tag
                    sc_results.append(r)
                    all_results.append(r)

        elapsed = time.time() - t_sc
        df_sc  = pd.DataFrame(sc_results)
        ok_n   = (df_sc["status"] == "ok").sum()
        conf_n = df_sc["status"].str.contains("conflict").sum()
        avg_ms = df_sc["duration_ms"].mean()
        tipos  = df_sc[df_sc["status"].str.contains("conflict")]["status"].value_counts().to_dict() if conf_n > 0 else {}

        icon     = "✅" if conf_n == 0 else "⚡"
        tipo_str = f" [{', '.join(f'{k}:{v}' for k,v in tipos.items())}]" if tipos else ""
        print(f"  {icon} {scenario_id:<15} ok={ok_n:>2}  conf={conf_n:>2}  "
              f"avg={avg_ms:>7.1f}ms  {elapsed:.1f}s{tipo_str}")

        all_summary.append({
            "mode": mode_tag, "scenario": scenario_id, "desc": desc,
            "expected": expected, "ok": int(ok_n), "conflicts": int(conf_n),
            "avg_ms": round(avg_ms, 1), "elapsed_s": round(elapsed, 1),
            "conflict_types": list(tipos.keys()),
        })

stop_workers()
print(f"\n✅ Completado. Total registros: {len(all_results)}")

In [ ]:
# ── COMPARATIVA FINAL ──────────────────────────────────────

df_sum = pd.DataFrame(all_summary)
df_all = pd.DataFrame(all_results)

# Pivot conflictos
pivot_conf = df_sum.pivot_table(
    index=["scenario", "desc", "expected"],
    columns="mode", values="conflicts", aggfunc="sum"
).fillna(0).astype(int).reset_index()
pivot_conf["expected_conflict"] = pivot_conf["expected"].str.startswith("⚡")
pivot_conf["expected_label"] = pivot_conf["expected_conflict"].map({True: "conflicto esperado", False: "sin conflicto esperado"})
pivot_conf["cumple_COW"] = pivot_conf.apply(lambda row: (row.get("COW", 0) > 0) == row["expected_conflict"], axis=1)
pivot_conf["cumple_MOR"] = pivot_conf.apply(lambda row: (row.get("MOR", 0) > 0) == row["expected_conflict"], axis=1)

print("=" * 70)
print("CONFLICTOS: COW vs MOR")
print("=" * 70)
print(pivot_conf[["scenario", "desc", "expected_label", "COW", "MOR", "cumple_COW", "cumple_MOR"]].to_string(index=False))

# Pivot tiempos
pivot_ms = df_sum.pivot_table(
    index=["scenario", "desc"],
    columns="mode", values="avg_ms", aggfunc="mean"
).round(1).reset_index()
if "COW" in pivot_ms.columns and "MOR" in pivot_ms.columns:
    pivot_ms["diff_ms"]    = (pivot_ms["MOR"] - pivot_ms["COW"]).round(1)
    pivot_ms["MOR_vs_COW"] = (pivot_ms["diff_ms"] / pivot_ms["COW"] * 100).round(1).astype(str) + "%"

print("\n" + "=" * 70)
print("TIEMPOS MEDIOS (ms): COW vs MOR")
print("=" * 70)
print(pivot_ms.to_string(index=False))

# Tipos de conflicto
print("\n" + "=" * 70)
print("TIPOS DE CONFLICTO POR MODO")
print("=" * 70)
conflicts = df_all[df_all["status"].str.contains("conflict", na=False)]
if not conflicts.empty:
    print(conflicts.groupby(["write_mode", "operation", "status"]).size().rename("count").to_string())

# Veredicto
print("\n" + "=" * 70)
print("VEREDICTO")
print("=" * 70)
for _, row in pivot_conf.iterrows():
    cow_c = int(row.get("COW", 0))
    mor_c = int(row.get("MOR", 0))
    esperaba_conflicto = bool(row["expected_conflict"])
    cow_icon = "❌" if cow_c>0 else "✅"
    mor_icon = "❌" if mor_c>0 else "✅"
    esperado_txt = "conflicto" if esperaba_conflicto else "sin conflicto"
    observado_txt = f"COW={cow_c} MOR={mor_c}"
    print(f"  COW{cow_icon} MOR{mor_icon}  {row['scenario']:<15} {row['desc']} | esperado: {esperado_txt} | observado: {observado_txt}")